# Pre-process Amazon Reviews 2023

Same steps as `process.ipynb`, using `package.preprocess` / `package.utils` instead of in-notebook defs.

# 0. Import & logging

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "package").is_dir())
sys.path.insert(0, str(ROOT))

from package.utils.path import processed_dir, raw_dir
from package.utils.log import experiment_log_path, setup_logging
from package.preprocess import (
    assign_idx,
    keep_first_filter,
    kcore_filter,
    leave_one_out_split,
    load_reviews_and_metadata,
    low_rating_filter,
    write_id_maps,
    write_products,
    write_simulator_jsonl,
    write_splits,
)

In [2]:
# "local" -> resource/local ; "recai" -> resource/recai  (raw always stays in resource/raw)
WORKSPACE = "local"

LOG_FILE = experiment_log_path("process", "amazon", workspace=WORKSPACE)
logger = setup_logging(name="process", log_file=LOG_FILE)

# 1. Configuration

In [3]:
SEED = 2024
CATEGORY = "All_Beauty"

RATING_THRESHOLD = 3.0
USER_K = 5
ITEM_K = 5

MAX_HISTORY_LEN = 10
MAX_TITLE_LEN = 50
MAX_DESCRIPTION_SENTENCES = 2
SIMULATOR_SAMPLE_N = 900

DATA_DIR = raw_dir(CATEGORY)
OUTPUT_DIR = processed_dir(WORKSPACE, CATEGORY)

print(f"workspace: {WORKSPACE}")
print(f"raw: {DATA_DIR}")
print(f"out: {OUTPUT_DIR}")
print(f"log: {LOG_FILE}")

workspace: local
raw: C:\Users\ph181\Documents\Repositories\agentic-rag-for-rcm-sys\resource\raw\All_Beauty
out: C:\Users\ph181\Documents\Repositories\agentic-rag-for-rcm-sys\resource\local\processed\All_Beauty
log: C:\Users\ph181\Documents\Repositories\agentic-rag-for-rcm-sys\resource\local\log\process\Exp2_amazon_20260922.log


# 2–3. Load, clean metadata, select columns

`load_reviews_and_metadata` already does the old cells 10–17: jsonl cache, drop missing titles, join description, take first category, rename `parent_asin` → `item_id`, keep reviews whose item is in meta.

In [4]:
review_df, meta_df = load_reviews_and_metadata(
    CATEGORY, logger, max_description_sentences=MAX_DESCRIPTION_SENTENCES
)
print(review_df.shape, meta_df.shape)
review_df.head()

2026-09-22 15:30:46 [INFO] process: Loading cached data from C:\Users\ph181\Documents\Repositories\agentic-rag-for-rcm-sys\resource\raw\All_Beauty\reviews.tsv
2026-09-22 15:30:51 [INFO] process: Loading cached data from C:\Users\ph181\Documents\Repositories\agentic-rag-for-rcm-sys\resource\raw\All_Beauty\meta.tsv
2026-09-22 15:30:55 [INFO] process: Shape of reviews: (701444, 4)
2026-09-22 15:30:55 [INFO] process: Shape of meta: (112578, 5)
(701444, 4) (112578, 5)


,user_id,item_id,rating,timestamp
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,1588687728923
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,1588615855070
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,1589665266052
3,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B09JS339BZ,1.0,1643393630220
4,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,5.0,1609322563534


# 4. Filter

In [5]:
data_df = keep_first_filter(review_df)
logger.info("After keep-first filter: %s", data_df.shape)

data_df = low_rating_filter(data_df, RATING_THRESHOLD)
logger.info("After rating filter: %s", data_df.shape)

data_df = kcore_filter(data_df, USER_K, ITEM_K)
logger.info("After k-core filter: %s", data_df.shape)

2026-09-22 15:30:57 [INFO] process: After keep-first filter: (693847, 4)
2026-09-22 15:30:57 [INFO] process: After rating filter: (550401, 4)
2026-09-22 15:30:57 [INFO] process: After k-core filter: (1888, 4)


# 5. Map user and item ids

In [6]:
data_df, user_map, item_map = assign_idx(data_df)
write_id_maps(item_map, user_map, CATEGORY, WORKSPACE)
logger.info("Saved id maps to %s", OUTPUT_DIR / "map.json")

2026-09-22 15:30:57 [INFO] process: Saved id maps to C:\Users\ph181\Documents\Repositories\agentic-rag-for-rcm-sys\resource\local\processed\All_Beauty\map.json


# 6. Leave-one-out split

In [7]:
df_train, df_valid, df_test, df_train_0 = leave_one_out_split(data_df)
write_splits(df_train, df_valid, df_test, df_train_0, CATEGORY, WORKSPACE)
logger.info(
    "Saved splits to %s (train=%d, valid=%d, test=%d, full_history=%d)",
    OUTPUT_DIR, len(df_train), len(df_valid), len(df_test), len(df_train_0),
)

2026-09-22 15:30:57 [INFO] process: Saved splits to C:\Users\ph181\Documents\Repositories\agentic-rag-for-rcm-sys\resource\local\processed\All_Beauty (train=1492, valid=198, test=198, full_history=1690)


# 7. Product table

In [8]:
saved_meta_df = write_products(meta_df, item_map, df_train_0, CATEGORY, WORKSPACE)
logger.info("Saved product table (%d items) to %s", len(saved_meta_df), OUTPUT_DIR)

2026-09-22 15:30:57 [INFO] process: Saved product table (280 items) to C:\Users\ph181\Documents\Repositories\agentic-rag-for-rcm-sys\resource\local\processed\All_Beauty


# 8. Simulator jsonl sample

In [9]:
simulator_path = write_simulator_jsonl(
    df_test,
    df_train_0,
    saved_meta_df,
    CATEGORY,
    WORKSPACE,
    sample_n=SIMULATOR_SAMPLE_N,
    seed=SEED,
    max_history_len=MAX_HISTORY_LEN,
    max_title_len=MAX_TITLE_LEN,
)
logger.info("Pipeline complete: %s", simulator_path)

2026-09-22 15:30:57 [INFO] process: Pipeline complete: C:\Users\ph181\Documents\Repositories\agentic-rag-for-rcm-sys\resource\local\processed\All_Beauty\simulator_test_data_198.jsonl
